# Full Pipeline: OMOP CDM → Custom Analytics CDM

Define a flat analytics mart as a YAML standard, map OMOP into it, and
run cohort / prevalence / age analytics on the result.

> **Prerequisites** — `pip install "portiere-health[polars]"`. This notebook is
> **fully offline**: no model downloads, no network calls, no API keys. It uses
> the bundled ICD-10-CM / LOINC / RxNorm vocabulary subset and BM25 lexical
> retrieval, and asserts the no-egress guarantee with `offline=True`.
> Every cell is deterministic — run top to bottom.

In [1]:
import json
import tempfile
from pathlib import Path

import polars as pl

import portiere
from portiere import __version__

WORK = Path(tempfile.mkdtemp(prefix="portiere_nb_"))
print(f"portiere v{__version__} — working dir: {WORK}")

portiere v0.4.0 — working dir: /var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_oatnwm0m


## Step 1 — The source: an existing OMOP CDM instance

You already converted to OMOP; now the analytics team wants a flat,
query-friendly mart. We synthesize two OMOP tables inline (in production,
export them from your CDM database).

In [2]:
import random

random.seed(11)
omop_dir = WORK / "omop_cdm"
omop_dir.mkdir(parents=True)

n = 80
person = pl.DataFrame({
    "person_id":        list(range(1, n + 1)),
    "gender_concept_id": [random.choice([8507, 8532]) for _ in range(n)],
    "year_of_birth":    [random.randint(1935, 2005) for _ in range(n)],
})
person.write_csv(omop_dir / "person.csv")

concepts = [201826, 320128, 4329847]  # T2DM, hypertension, MI (SNOMED-standard ids)
condition = pl.DataFrame({
    "condition_occurrence_id": list(range(1, n * 3 + 1)),
    "person_id":            [random.randint(1, n) for _ in range(n * 3)],
    "condition_concept_id": [random.choice(concepts) for _ in range(n * 3)],
    "condition_start_date": [f"202{random.randint(0, 4)}-{random.randint(1, 12):02d}-01"
                             for _ in range(n * 3)],
})
condition.write_csv(omop_dir / "condition_occurrence.csv")
print(person.head(3)); print(condition.head(3))

shape: (3, 3)
┌───────────┬───────────────────┬───────────────┐
│ person_id ┆ gender_concept_id ┆ year_of_birth │
│ ---       ┆ ---               ┆ ---           │
│ i64       ┆ i64               ┆ i64           │
╞═══════════╪═══════════════════╪═══════════════╡
│ 1         ┆ 8532              ┆ 1984          │
│ 2         ┆ 8532              ┆ 1949          │
│ 3         ┆ 8532              ┆ 1985          │
└───────────┴───────────────────┴───────────────┘
shape: (3, 4)
┌─────────────────────────┬───────────┬──────────────────────┬──────────────────────┐
│ condition_occurrence_id ┆ person_id ┆ condition_concept_id ┆ condition_start_date │
│ ---                     ┆ ---       ┆ ---                  ┆ ---                  │
│ i64                     ┆ i64       ┆ i64                  ┆ str                  │
╞═════════════════════════╪═══════════╪══════════════════════╪══════════════════════╡
│ 1                       ┆ 48        ┆ 4329847              ┆ 2024-03-01           │
│ 2   

## Step 2 — Define the custom analytics CDM as YAML

A custom standard is a YAML file: entities, typed fields, and
`source_patterns` (fast-path column matching; values are target **field
names** within the entity). One modeling rule to know: patterns are global
per standard, so a source column shared by several tables (`person_id`) must
be claimed by exactly one entity — which is why the mart below is split into
one standard per extraction unit. Full reference:
`docs/documentations/20-multi-standard-support.md`.

In [3]:
# NOTE: source_patterns are GLOBAL per standard (pattern -> one entity.field),
# so a column name shared by several source tables (person_id here) must not
# be claimed by two entities of the same standard. The clean pattern is one
# mart standard per extraction unit — patients and events below.
PATIENTS_YAML = """
name: "patients_mart_v1"
version: "1.0"
standard_type: "relational"
organization: "Research Analytics"
description: "Patient-level mart for cohort selection"

entities:
  patient_summary:
    description: "One row per patient with demographics"
    fields:
      subject_id:
        type: integer
        required: true
        description: "Patient identifier carried from OMOP person_id"
      gender_code:
        type: integer
        required: false
        description: "Gender concept carried from OMOP gender_concept_id"
      birth_year:
        type: integer
        required: true
        description: "Year of birth for age computation"
    source_patterns:
      person_id: "subject_id"
      gender_concept_id: "gender_code"
      year_of_birth: "birth_year"

default_entity: "patient_summary"
default_field: "subject_id"
"""

EVENTS_YAML = """
name: "events_mart_v1"
version: "1.0"
standard_type: "relational"
organization: "Research Analytics"
description: "Long-format clinical events for analytics"

entities:
  event_flat:
    description: "One row per clinical event"
    fields:
      subject_id:
        type: integer
        required: true
        description: "Patient identifier linking to patient_summary"
      event_concept:
        type: integer
        required: true
        description: "Condition concept carried from OMOP"
      event_date:
        type: date
        required: true
        description: "Event date carried from condition_start_date"
    source_patterns:
      person_id: "subject_id"
      condition_concept_id: "event_concept"
      condition_start_date: "event_date"

default_entity: "event_flat"
default_field: "subject_id"
"""

patients_yaml = WORK / "patients_mart_v1.yaml"
patients_yaml.write_text(PATIENTS_YAML)
events_yaml = WORK / "events_mart_v1.yaml"
events_yaml.write_text(EVENTS_YAML)
print(patients_yaml.name, "and", events_yaml.name, "written")

patients_mart_v1.yaml and events_mart_v1.yaml written


In [4]:
from portiere.models.target_model import get_target_model

model = get_target_model(f"custom:{patients_yaml}")
schema = model.get_schema()
for entity, fields in schema.items():
    print(f"  {entity}: {list(fields)}")
print()
print(model.generate_ddl()[:400])

  patient_summary: ['subject_id', 'gender_code', 'birth_year']

CREATE TABLE patient_summary (
    subject_id INTEGER,
    gender_code INTEGER,
    birth_year INTEGER
);


## Step 3 — Map OMOP → the custom mart

`target_model="custom:<path>"` points the standardize pipeline at your mart.
The `source_patterns` make OMOP column names first-class matches — mappings
arrive at high confidence without any model download.

In [5]:
# Offline configuration: BM25 retrieval over the bundled vocabulary subset,
# no embedding model (nothing to download), no reranker, and offline=True —
# construction FAILS if any provider could send data off this machine.
from portiere._demo_data import vocabulary_dir
from portiere.config import (
    EmbeddingConfig,
    KnowledgeLayerConfig,
    PortiereConfig,
    RerankerConfig,
)
from portiere.knowledge import build_knowledge_layer

knowledge_paths = build_knowledge_layer(
    athena_path=str(vocabulary_dir()),
    output_path=str(WORK / "knowledge_index"),
    backend="bm25s",
    vocabularies=["ICD10CM", "LOINC", "RxNorm"],
)

config = PortiereConfig(
    local_project_dir=WORK / "project",
    knowledge_layer=KnowledgeLayerConfig(backend="bm25s", **knowledge_paths),
    embedding=EmbeddingConfig(provider="none"),
    reranker=RerankerConfig(provider="none", model=""),
    offline=True,  # the executable no-egress guarantee (v0.4.0)
)
print("offline mode asserted:", config.offline)

2026-07-24 10:11:13 [info     ] athena.loading_concepts        path=/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/src/portiere/_demo_data/vocabulary/CONCEPT.csv


2026-07-24 10:11:13 [info     ] athena.concepts_loaded         count=27 vocabularies=['ICD10CM', 'LOINC', 'RxNorm']


2026-07-24 10:11:13 [info     ] athena.bm25s_built             count=27 path=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_oatnwm0m/knowledge_index/concepts.json


2026-07-24 10:11:13 [info     ] athena.knowledge_layer_built   backend=bm25s concepts=27 paths={'bm25s_corpus_path': '/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_oatnwm0m/knowledge_index/concepts.json'}


offline mode asserted: True


In [6]:
proj_patients = portiere.init(
    name="omop-to-patients-mart",
    target_model=f"custom:{patients_yaml}",
    vocabularies=["ICD10CM"],
    config=config,
)

src_person = proj_patients.add_source(str(omop_dir / "person.csv"), name="person")
schema_map_p = proj_patients.map_schema(src_person)
for item in schema_map_p.items:
    print(f"  {item.source_column:22s} -> "
          f"{(item.effective_target_table or '?'):18s}."
          f"{(item.effective_target_column or '?'):16s} conf={item.confidence:.2f} "
          f"[{item.status}]")

2026-07-24 10:11:13 [info     ] PolarsEngine initialized      


2026-07-24 10:11:13 [debug    ] local_storage.initialized      base_dir=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_oatnwm0m/project


2026-07-24 10:11:13 [info     ] local_storage.project_created  name=omop-to-patients-mart project_id=0aad49f2-5b8e-492b-a02a-1a5d4a126d80


2026-07-24 10:11:13 [info     ] project.source_added           project=omop-to-patients-mart source=person


2026-07-24 10:11:13 [warning  ] manifest.git_dirty             hint='Working tree has uncommitted changes; manifest records git_dirty=true' project_root=/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/docs/notebooks_examples


...

2026-07-24 10:11:17 [info     ] local_schema_mapper.initialized standard=patients_mart_v1


2026-07-24 10:11:17 [info     ] Stage 2 complete               auto=3 review=0 total=3


2026-07-24 10:11:17 [info     ] local_storage.schema_mapping_saved items_count=3 project=omop-to-patients-mart


2026-07-24 10:11:17 [info     ] project.schema_mapped          auto=3 project=omop-to-patients-mart total=3


  person_id              -> patient_summary   .subject_id       conf=0.95 [MappingStatus.AUTO_ACCEPTED]
  gender_concept_id      -> patient_summary   .gender_code      conf=0.95 [MappingStatus.AUTO_ACCEPTED]
  year_of_birth          -> patient_summary   .birth_year       conf=0.95 [MappingStatus.AUTO_ACCEPTED]


In [7]:
etl_out_p = WORK / "mart" / "patients_run"
proj_patients.run_etl(src_person, output_dir=str(etl_out_p),
                      schema_mapping=schema_map_p)

proj_events = portiere.init(
    name="omop-to-events-mart",
    target_model=f"custom:{events_yaml}",
    vocabularies=["ICD10CM"],
    config=config,
)
src_cond = proj_events.add_source(str(omop_dir / "condition_occurrence.csv"),
                                  name="condition_occurrence")
schema_map_c = proj_events.map_schema(src_cond)
for item in schema_map_c.items:
    print(f"  {item.source_column:24s} -> "
          f"{(item.effective_target_table or '?'):12s}."
          f"{(item.effective_target_column or '?'):14s} [{item.status}]")
etl_out_c = WORK / "mart" / "events_run"
proj_events.run_etl(src_cond, output_dir=str(etl_out_c),
                    schema_mapping=schema_map_c)

mart_files = sorted((WORK / "mart").rglob("*.csv"))
for f in mart_files:
    print("  ", f.relative_to(WORK / "mart"))

2026-07-24 10:11:17 [info     ] Reading source                 format=csv path=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_oatnwm0m/omop_cdm/person.csv


2026-07-24 10:11:17 [info     ] Processing table               columns=3 table=patient_summary


2026-07-24 10:11:17 [info     ] ETL complete                   duration=0.003039 success=True


2026-07-24 10:11:17 [info     ] project.etl_complete           output_dir=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_oatnwm0m/mart/patients_run project=omop-to-patients-mart


2026-07-24 10:11:17 [info     ] PolarsEngine initialized      


...

  condition_occurrence_id  -> event_flat  .event_date     [MappingStatus.UNMAPPED]
  person_id                -> event_flat  .subject_id     [MappingStatus.AUTO_ACCEPTED]
  condition_concept_id     -> event_flat  .event_concept  [MappingStatus.AUTO_ACCEPTED]
  condition_start_date     -> event_flat  .event_date     [MappingStatus.AUTO_ACCEPTED]
2026-07-24 10:11:17 [info     ] Reading source                 format=csv path=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_oatnwm0m/omop_cdm/condition_occurrence.csv


2026-07-24 10:11:17 [info     ] Processing table               columns=3 table=event_flat


2026-07-24 10:11:17 [info     ] ETL complete                   duration=0.002201 success=True


2026-07-24 10:11:17 [info     ] project.etl_complete           output_dir=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_oatnwm0m/mart/events_run project=omop-to-events-mart


   events_run/event_flat.csv
   patients_run/patient_summary.csv


## Step 4 — Advanced analytics on the mart

The point of the flat mart: cohort queries become one-liners. All polars,
all local.

In [8]:
ps_files = [f for f in mart_files if f.name == "patient_summary.csv"]
ev_files = [f for f in mart_files if f.name == "event_flat.csv"]
patients_df = pl.read_csv(ps_files[0])
events_df = pl.read_csv(ev_files[0])
print(f"mart: {patients_df.height} patients ({patients_df.columns}), "
      f"{events_df.height} events ({events_df.columns})")

mart: 80 patients (['subject_id', 'gender_code', 'birth_year']), 240 events (['subject_id', 'event_concept', 'event_date'])


In [9]:
# Cohort: patients born before 1970 with >= 2 distinct condition concepts
cohort = (
    events_df.group_by("subject_id")
    .agg(pl.col("event_concept").n_unique().alias("n_conditions"))
    .join(patients_df, on="subject_id")
    .filter((pl.col("birth_year") < 1970) & (pl.col("n_conditions") >= 2))
)
print(f"cohort size: {cohort.height}")
print(cohort.head(5))

cohort size: 33
shape: (5, 4)
┌────────────┬──────────────┬─────────────┬────────────┐
│ subject_id ┆ n_conditions ┆ gender_code ┆ birth_year │
│ ---        ┆ ---          ┆ ---         ┆ ---        │
│ i64        ┆ u32          ┆ i64         ┆ i64        │
╞════════════╪══════════════╪═════════════╪════════════╡
│ 2          ┆ 3            ┆ 8532        ┆ 1949       │
│ 5          ┆ 3            ┆ 8507        ┆ 1962       │
│ 6          ┆ 3            ┆ 8532        ┆ 1935       │
│ 9          ┆ 2            ┆ 8532        ┆ 1937       │
│ 10         ┆ 3            ┆ 8532        ┆ 1961       │
└────────────┴──────────────┴─────────────┴────────────┘


In [10]:
# Prevalence per condition concept, by gender
prevalence = (
    events_df.join(patients_df, on="subject_id")
    .group_by(["event_concept", "gender_code"])
    .agg(pl.col("subject_id").n_unique().alias("n_patients"))
    .sort(["event_concept", "gender_code"])
)
print(prevalence)

shape: (6, 3)
┌───────────────┬─────────────┬────────────┐
│ event_concept ┆ gender_code ┆ n_patients │
│ ---           ┆ ---         ┆ ---        │
│ i64           ┆ i64         ┆ u32        │
╞═══════════════╪═════════════╪════════════╡
│ 201826        ┆ 8507        ┆ 30         │
│ 201826        ┆ 8532        ┆ 26         │
│ 320128        ┆ 8507        ┆ 31         │
│ 320128        ┆ 8532        ┆ 17         │
│ 4329847       ┆ 8507        ┆ 29         │
│ 4329847       ┆ 8532        ┆ 19         │
└───────────────┴─────────────┴────────────┘


In [11]:
# Age distribution of the full mart (2026 reference year)
ages = patients_df.with_columns((2026 - pl.col("birth_year")).alias("age"))
print(ages.select(
    pl.col("age").min().alias("min"),
    pl.col("age").mean().round(1).alias("mean"),
    pl.col("age").max().alias("max"),
))

shape: (1, 3)
┌─────┬──────┬─────┐
│ min ┆ mean ┆ max │
│ --- ┆ ---  ┆ --- │
│ i64 ┆ f64  ┆ i64 │
╞═════╪══════╪═════╡
│ 21  ┆ 60.6 ┆ 91  │
└─────┴──────┴─────┘
